# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset overview
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In `mlcroissant`, you can access record set and field `@id`s and their structure as follows. Note that each entity is identified by its `@id`.

In [ ]:
# List available record sets and their fields, showing @id for each
print("\nAvailable Record Sets:")
record_sets = list(dataset.record_sets())

record_set_ids = []

for rs in record_sets:
    print(f"- Record set name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("   Fields (columns):")
    for field in rs.fields:
        print(f"    - {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s as shown above.

In [ ]:
# Extract records for each record set and build DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded record set {rs_id} (rows: {len(records)})")
    else:
        print(f"Warning: Record set {rs_id} is empty or could not be loaded.")

# For demonstration, pick the first non-empty record set for EDA
target_rs_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes and not dataframes[rs_id].empty:
        target_rs_id = rs_id
        break
if target_rs_id:
    print(f"\nColumn names in DataFrame for record set {target_rs_id}:")
    print(dataframes[target_rs_id].columns.tolist())
    display(dataframes[target_rs_id].head())
else:
    print("No data available for any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This often involves removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Example: select a numeric field for analysis
# Let's try to automatically pick a numeric field if present
numeric_field = None
df = dataframes[target_rs_id]

for col in df.columns:
    # Try to convert to float, see if that works
    try:
        df[col] = pd.to_numeric(df[col])
        # If it has more than 10 unique values, treat as continuous
        if df[col].dtype.kind in 'fi' and df[col].nunique() > 10:
            numeric_field = col
            break
    except:
        continue
if numeric_field is None:
    raise ValueError('No numeric field found for analysis.')
print(f"Using numeric field: '{numeric_field}' for EDA.")

# Apply threshold filtering
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a non-numeric field
group_field = None
for col in df.columns:
    if df[col].dtype == object and df[col].nunique() < 20 and col != numeric_field:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"\nGrouped data by '{group_field}':")
    display(grouped_df.head())
else:
    print('No suitable group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='teal')
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If grouping field exists, plot group means
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field, palette='mako')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
Summarize your findings and observations from this dataset exploration.